In [ ]:
import os

import numpy as np
import pandas as pd
import wandb
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerModelCardData,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.sentence_transformer.losses import (
    CachedGISTEmbedLoss,
    CachedMultipleNegativesRankingLoss,
    MultipleNegativesRankingLoss,
)
from sentence_transformers.sentence_transformer.training_args import (
    BatchSamplers,
)


os.environ["WANDB_PROJECT"] = "HKHS-Code-Search"
os.environ["WANDB_LOG_MODEL"] = "true"
os.environ["WANDB_WATCH"]="false"

wandb.login()

In [ ]:
col = ["chapter", "heading", "subheading", "hkhs", "desc", "field"]
corpora = pd.read_parquet("../data/processed/corpora.parquet", columns=col)
synthetic = pd.read_parquet("../data/processed/synthetic.parquet", columns=col)

hkhs = pd.concat([corpora, synthetic], ignore_index=True)

##### Variant A: legal description (positive) and another subheading of the same heading (negative)

In [ ]:
hs2_hs4_desc = {
    heading: list(zip(group["subheading"], group["desc"]))
    for heading, group in hkhs.query("field == 'description'") \
        [["heading", "subheading", "desc"]].groupby("heading", sort=False)
}

hs6_desc = hkhs.query("field == 'description'") \
    .groupby("subheading", sort=False)["desc"] \
    .agg(list).to_dict()

hs8_desc = hkhs.query("field == 'description'") \
    .set_index("hkhs")["desc"] \
    .to_dict()

rng = np.random.default_rng(seed=42)

data = []
for rw in hkhs.query("field in ['commodity', 'synthetic']").itertuples():
    for subheading, desc in hs2_hs4_desc[rw.heading]:
        match rw.field, rw.subheading != subheading:
            case "commodity", True:
                pos = hs8_desc[rw.hkhs]
            case "synthetic", True:
                pos = rng.choice(hs6_desc[rw.subheading])
            case _:
                continue
        data.append({"anchor": rw.desc, "positive": pos, "negative": desc})

df = pd.DataFrame(data)
df.query("anchor != positive", inplace=True)
df.reset_index(drop=True, inplace=True)

train = Dataset.from_pandas(df)

In [ ]:
model_root = "../checkpoints"
model_card = "BAAI/bge-small-en-v1.5"
model_exp = "bge-small-en-v1.5-ft-hs"
model_tag = "v0.1"

model = SentenceTransformer(
    model_card,
    device="cuda",
    model_card_data=SentenceTransformerModelCardData(
        language="en",
        license="mit",
        model_name="BGE small English v1.5 fine-tuned on HKHS descriptions",
    ),
)

loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir="/".join([model_root, model_exp]),
    num_train_epochs=1,
    per_device_train_batch_size=64,
    learning_rate=2e-5,
    warmup_steps=0.1,
    fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy="no",
    logging_steps=1_000,
    save_steps=1_000,
    report_to="wandb",
    run_name="/".join([model_root, model_exp, model_tag]),
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train,
    loss=loss,
)
trainer.train()
wandb.finish()

model.save("/".join([model_root, model_exp, model_tag]))

In [ ]:
model_root = "../checkpoints"
model_card = "BAAI/bge-small-en-v1.5"
model_exp = "bge-small-en-v1.5-ft-hs"
model_tag = "v0.2"

model = SentenceTransformer(
    model_card,
    device="cuda",
    model_card_data=SentenceTransformerModelCardData(
        language="en",
        license="mit",
        model_name="BGE small English v1.5 fine-tuned on HKHS descriptions",
    ),
)

loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir="/".join([model_root, model_exp]),
    num_train_epochs=2,
    per_device_train_batch_size=64,
    learning_rate=2e-5,
    warmup_steps=0.1,
    fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy="no",
    logging_steps=1_000,
    save_steps=1_000,
    report_to="wandb",
    run_name="/".join([model_root, model_exp, model_tag]),
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train,
    loss=loss,
)
trainer.train()
wandb.finish()

model.save("/".join([model_root, model_exp, model_tag]))

##### Variant B: legal description (positive) and random subheading of the same heading (negative)

In [ ]:
rng = np.random.default_rng(seed=42)

data = []
for rw in hkhs.query("field in ['commodity', 'synthetic']").itertuples():
    negs = [d for s, d in hs2_hs4_desc[rw.heading] if s != rw.subheading]
    if not negs:
        continue
    match rw.field:
        case "commodity":
            pos = hs8_desc[rw.hkhs]
        case "synthetic":
            pos = rng.choice(hs6_desc[rw.subheading])
    data.append({
        "anchor": rw.desc,
        "positive": pos,
        "negative": rng.choice(negs),
    })

df = pd.DataFrame(data)
df.query("anchor != positive", inplace=True)
df.reset_index(drop=True, inplace=True)

train = Dataset.from_pandas(df)

In [ ]:
model_root = "../checkpoints"
model_card = "BAAI/bge-small-en-v1.5"
model_exp = "bge-small-en-v1.5-ft-hs"
model_tag = "v0.3"

model = SentenceTransformer(
    model_card,
    device="cuda",
    model_card_data=SentenceTransformerModelCardData(
        language="en",
        license="mit",
        model_name="BGE small English v1.5 fine-tuned on HKHS descriptions",
    ),
)

loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir="/".join([model_root, model_exp]),
    num_train_epochs=2,
    per_device_train_batch_size=64,
    learning_rate=1e-5,
    warmup_steps=0.1,
    fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy="no",
    logging_steps=300,
    save_steps=300,
    report_to="wandb",
    run_name="/".join([model_root, model_exp, model_tag]),
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train,
    loss=loss,
)
trainer.train()
wandb.finish()

model.save("/".join([model_root, model_exp, model_tag]))

In [ ]:
model_root = "../checkpoints"
model_card = "BAAI/bge-small-en-v1.5"
model_exp = "bge-small-en-v1.5-ft-hs"
model_tag = "v0.4"

model = SentenceTransformer(
    model_card,
    device="cuda",
    model_card_data=SentenceTransformerModelCardData(
        language="en",
        license="mit",
        model_name="BGE small English v1.5 fine-tuned on HKHS descriptions",
    ),
)

model.max_seq_length = 64
loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=32)

args = SentenceTransformerTrainingArguments(
    output_dir="/".join([model_root, model_exp]),
    num_train_epochs=2,
    per_device_train_batch_size=256,
    learning_rate=2e-5,
    warmup_steps=0.1,
    fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy="no",
    logging_steps=20,
    save_steps=300,
    report_to="wandb",
    run_name="/".join([model_root, model_exp, model_tag]),
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train,
    loss=loss,
)
trainer.train()
wandb.finish()

model.save("/".join([model_root, model_exp, model_tag]))

In [ ]:
model_root = "../checkpoints"
model_card = "BAAI/bge-small-en-v1.5"
model_exp = "bge-small-en-v1.5-ft-hs"
model_tag = "v0.5"

model = SentenceTransformer(
    model_card,
    device="cuda",
    model_card_data=SentenceTransformerModelCardData(
        language="en",
        license="mit",
        model_name="BGE small English v1.5 fine-tuned on HKHS descriptions",
    ),
)

model.max_seq_length = 64
loss = CachedGISTEmbedLoss(
    model,
    guide=SentenceTransformer(model_card),
    mini_batch_size=32,
)

args = SentenceTransformerTrainingArguments(
    output_dir="/".join([model_root, model_exp]),
    num_train_epochs=2,
    per_device_train_batch_size=256,
    learning_rate=2e-5,
    warmup_steps=0.1,
    fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy="no",
    logging_steps=20,
    save_steps=300,
    report_to="wandb",
    run_name="/".join([model_root, model_exp, model_tag]),
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train,
    loss=loss,
)
trainer.train()
wandb.finish()

model.save("/".join([model_root, model_exp, model_tag]))